# Event context: where was THA, and when did it cross a boundary?

`mango.timeline(sc, start, stop)` returns every MANGO sample of one spacecraft over a window
of up to 31 days, across all regions, with a `region` column. Here: THEMIS-A (`THA`) on
9 and 10 August 2008.

```{note}
These pages run on a small real-data sample served locally (dataset version
`2026.0-docs-sample`); this two-day window of THA is part of it.
```

In [ ]:
import matplotlib.pyplot as plt
import polars as pl

import space_mango as mango

SC, START, STOP = "THA", "2008-08-09T00:00:00", "2008-08-11T00:00:00"


In [ ]:
t = mango.timeline(SC, START, STOP, columns=["Bx", "By", "Bz", "Np"])
t

`to_intervals()` groups consecutive samples into intervals: a new interval starts when
the region changes or when two samples are more than `max_gap` apart (30 s by default).
`stop` is the last sample of the interval.

In [ ]:
iv = t.to_intervals()
iv

The boundary crossings, as classified by MANGO, are the intervals whose region differs
from the previous one; `start` is then the first sample in the new region.
This also counts a region change across a data gap and short (1 to 3 samples) flickers between regions; add a gap check or a minimum duration if you reuse it on other windows.

In [ ]:
crossings = iv.filter(pl.col("region") != pl.col("region").shift())
crossings

## Field and density with the region classification

Each interval is shaded with the color of its region; unshaded stretches have no MANGO
sample.

In [ ]:
REGION_COLORS = {"magnetosphere": "tab:blue", "magnetosheath": "tab:orange", "solar_wind": "tab:green"}
meta = t.metadata
df = t.to_polars().with_columns(B=(pl.col("Bx") ** 2 + pl.col("By") ** 2 + pl.col("Bz") ** 2).sqrt())

fig, (ax_b, ax_n) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax_b.plot(df["Time"].to_numpy(), df["B"].to_numpy(), ".", color="black", markersize=1)
ax_n.plot(df["Time"].to_numpy(), df["Np"].to_numpy(), ".", color="black", markersize=1)
for ax in (ax_b, ax_n):
    for row in iv.iter_rows(named=True):
        ax.axvspan(row["start"], row["stop"], color=REGION_COLORS[row["region"]], alpha=0.3,
                   linewidth=0, label=row["region"])

ax_b.set_ylabel(f"|B| ({meta['Bx']['unit']})")
ax_n.set_ylabel(f"Np ({meta['Np']['unit']})")
ax_n.set_yscale("log")
ax_n.set_xlabel("Time (UTC)")
ax_b.set_title(f"{SC}, {START} to {STOP}")
handles, labels = ax_b.get_legend_handles_labels()
by_region = dict(zip(labels, handles, strict=True))  # one legend entry per region
shown = [region for region in REGION_COLORS if region in by_region]
ax_b.legend([by_region[region] for region in shown], shown, loc="upper right", ncols=3)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## Next steps

The intervals are ordinary polars rows, so they can drive further analysis. For example, with
[speasy](https://github.com/SciQLop/speasy) you could fetch higher-resolution data for each
magnetosheath interval (not run here):

```python
import speasy as spz

for row in iv.filter(pl.col("region") == "magnetosheath").iter_rows(named=True):
    data = spz.get_data(product, row["start"], row["stop"])
```

where `product` is the speasy product you need (browse `spz.inventories.data_tree` to find it).
`mango.timeline` is limited to 31 days per call; loop over windows for longer periods.